# HR assistant

An engineering employee asks what the parental leave policy is in Germany.
The agent calls the gateway MCP server as `employee_eng`. It should search,
then cite the policy source, version, and sync time.

The same question as `outsider` is empty. The agent must say it found nothing
and must not guess a policy.

This notebook needs `OPENROUTER_API_KEY` and a model that can call tools.
`openai/gpt-4o-mini` works. A small model usually does not. Set
`OPENROUTER_MODEL`. LangSmith stays off unless `LANGSMITH_API_KEY` is set.

The gateway must be up, and the Germany policy must already be ingested.


In [1]:
import os
import sys
from pathlib import Path

here = Path.cwd()
if not (here / "agents_demo.py").exists():
    here = here / "notebooks"
if str(here) not in sys.path:
    sys.path.insert(0, str(here))

import agents_demo

print("model", os.environ.get("OPENROUTER_MODEL", ""))


model anthropic/claude-haiku-5.5


In [2]:
QUESTION = "What is our parental leave policy in Germany?"

agent = agents_demo.build_agent("employee_eng", agents_demo.HR_ROLE)
report = agents_demo.run(agent, QUESTION)
agents_demo.print_trace(report["trace"])
print(report["answer"])

hits = []
for row in report["trace"]:
    payload = row["payload"]
    if isinstance(payload, dict):
        hits.extend(payload.get("results") or [])
matched = [hit for hit in hits if hit.get("object_id") == "pol-parental-leave-de"]
if not matched:
    raise SystemExit(f"HR agent missed the Germany policy: {hits}")
hit = matched[0]
print(
    "citation",
    hit["source_id"],
    hit["object_id"],
    "v" + str(hit["version"]),
    "synced",
    hit["synced_at"],
)
if not hit.get("version") or not hit.get("synced_at"):
    raise SystemExit(f"citation is incomplete: {hit}")
if "pol-parental-leave-de" not in report["answer"]:
    raise SystemExit(f"answer did not cite the policy: {report['answer']}")


search_knowledge {'query': 'parental leave policy Germany'} -> 1
Our HR knowledge base has a Germany parental leave policy, but the excerpt I can read covers only the statutory leave entitlement (Elternzeit):

- Each parent is entitled to up to three years of unpaid parental leave per child.
- Up to 24 months of that can be taken between the child's third and eighth birthday, with the employer's agreement.
- Employment is protected from the moment you give notice, which can be at most eight weeks before the leave starts for a child under three, until the leave ends.

The excerpt doesn't mention parental allowance (Elterngeld) or any company top-up, so I can't tell you about those. Check with HR for the rest of the policy.

Source: `hr`, object `pol-parental-leave-de` (policy), version 1, last synced 2026-10-12T10:50:08.928569+00:00.
citation hr pol-parental-leave-de v1 synced 2026-10-12T10:50:08.928569+00:00


In [3]:
denied_agent = agents_demo.build_agent("outsider", agents_demo.HR_ROLE)
denied = agents_demo.run(denied_agent, QUESTION)
agents_demo.print_trace(denied["trace"])
print(denied["answer"])

searches = [row for row in denied["trace"] if row["tool"] == "search_knowledge"]
if not searches or any((row["result_count"] or 0) > 0 for row in searches):
    raise SystemExit(f"outsider should see no policy: {denied['trace']}")
if "pol-parental-leave-de" in denied["answer"]:
    raise SystemExit(denied["answer"])
print("outsider saw nothing")


search_knowledge {'query': 'parental leave policy Germany'} -> 0
search_knowledge {'query': 'Elternzeit parental leave Germany employees'} -> 0
search_knowledge {'query': 'parental leave'} -> 0
I searched our knowledge base three ways (“parental leave policy Germany,” “Elternzeit parental leave Germany employees,” and “parental leave”), and none of the results were available to me. I found no parental leave policy document for Germany, or any general parental leave policy, that I'm permitted to read.

I'm not able to give you the policy from general knowledge, since that could be inaccurate for your company. Some next steps:

- Check with HR operations or your German HR/people team, who can confirm the current policy.
- Ask whether a Germany-specific employee handbook or local addendum exists, since it may be filed under a different title.
- If you have the document, share the relevant section and I can help you interpret it, as long as it's in a source I can access.
outsider saw nothi